# Mortgage Default Risk Prediction

### Retail Loan Mortgage Approval — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of retail lending and mortgage approval terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the retail lending and mortgage approval problem and why the chosen classification approach suits this banking workflow.
- Implement an end-to-end classification pipeline on synthetic data and evaluate performance with domain-appropriate metrics.
- Assess whether the model is operationally viable, considering error costs, fairness, and the limitations of synthetic data.

**Estimated time:** 35–45 minutes

**Collection context:** Mortgage risk, automated underwriting, appraisal valuation, and loan approval processes in retail banking.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Banks lend hundreds of billions in mortgages. Even a small error in predicting who will default can lead to massive losses (as seen in the 2008 financial crisis). AI helps banks price risk more accurately.

**Input data used:** Credit Score (FICO), Loan-to-Value (LTV) ratio, Income, Debt-to-Income (DTI) ratio.

**What we predict:** Binary default label (`is_default`).

**ML method used:** Logistic Regression (the industry standard for credit risk).

**Learning goal:** Understand how 'Risk Drivers' like low credit scores and high loan amounts increase the chance of default.

**Primary stakeholders:** mortgage officers, underwriters, credit risk managers, and compliance teams.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Mortgage Data

We create 5,000 synthetic mortgage applications.

In [ ]:
n_apps = 5000
credit_score = RNG.integers(500, 850, n_apps)
ltv = RNG.uniform(0.6, 1.1, n_apps) # Loan-to-Value (80% is standard, >100% is risky)
dti = RNG.uniform(0.1, 0.6, n_apps) # Debt-to-Income ratio

# Logic for Default (Hidden patterns)
# Default is likely if Credit Score is low, LTV is high, and DTI is high.
logit = (
    -0.01 * (credit_score - 700) + 
    5.0 * (ltv - 0.8) + 
    3.0 * (dti - 0.3) + 
    RNG.normal(0, 0.5, n_apps)
)
prob_default = 1 / (1 + np.exp(-logit))
is_default = (prob_default > 0.5).astype(int)

df = pd.DataFrame({
    'credit_score': credit_score,
    'ltv': ltv,
    'dti': dti,
    'is_default': is_default
})

print(f"Dataset created. Default rate: {df['is_default'].mean():.1%}")

## Step 4 — Exploratory Data Analysis

EDA (Exploratory Data Analysis)

Let's visualize how Credit Score and LTV relate to defaults.

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(data=df, x='credit_score', hue='is_default', multiple='stack')
plt.title('Default Distribution by Credit Score')

plt.subplot(1, 2, 2)
sns.boxplot(x='is_default', y='ltv', data=df)
plt.title('LTV Ratio: Normal vs Default')

plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Distribution plot, histogram titled "Default Distribution by Credit Score" and "LTV Ratio: Normal vs Default". The chart highlights distributional differences between groups that inform the modelling approach.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

In [ ]:
X = df.drop('is_default', axis=1)
y = df['is_default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the majority class ---
baseline_clf = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
baseline_clf.fit(X_train, y_train)
baseline_pred = baseline_clf.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_pred)
print(f"Baseline accuracy (majority-class): {baseline_acc:.3f}")
print(f"Any useful model must beat this {baseline_acc:.3f} floor.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [ ]:
print("Confusion Matrix:")
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()
plt.close()

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

**Alt-text:** Heatmap with Predicted on the x-axis and Actual on the y-axis. The heatmap reveals correlation strength and direction between variables.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

1. **Impact of Credit Score:** The negative coefficient for credit score means as the score goes up, the risk of default goes down.
2. **Impact of LTV:** Higher LTV (meaning the loan is almost the same value as the house) significantly increases risk.
3. **Bank Policy:** A bank might set a threshold: "Only approve if predicted probability of default is < 5%."

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: score representative cases from the test set ---
print("Operational demonstration — model decision support:\n")
proba = model.predict_proba(X_test)[:, 1]
low_idx  = int(np.argmin(proba))
high_idx = int(np.argmax(proba))
mid_idx  = int(np.argsort(proba)[len(proba) // 2])

for label, idx in [("Low-risk", low_idx), ("Medium-risk", mid_idx), ("High-risk", high_idx)]:
    row = X_test.iloc[idx]
    prob = proba[idx]
    print(f"{label} case  predicted probability: {prob:.1%}")
    print(f"  Features: {dict(row.round(2))}")
    print()

print("A decision-maker would combine these scores with business rules and domain judgement.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end retail lending and mortgage approval workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Mortgage models must comply with fair lending laws and avoid discriminatory approval patterns.
- Automated underwriting can disadvantage applicants from historically underserved communities.
- Transparency in loan decisions is a regulatory requirement, not an optional feature.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in retail lending and mortgage approval?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.